In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [2]:
TRAIN_DIR = Path("../data/raw/GTSRB/Final_Training/Images")
TEST_DIR = Path("../data/raw/GTSRB/Final_Test/Images")
TEST_LABELS = Path("../data/raw/GT-final_test.csv")

print(TRAIN_DIR.exists())
print(TEST_DIR.exists())
print(TEST_LABELS.exists())

True
True
True


In [3]:
IMG_SIZE = (96, 96)
NUM_CLASSES = 43

In [4]:
def load_training_data(train_dir, img_size):
    images = []
    labels = []

    for class_id in range(43):
        class_path = train_dir / format(class_id, "05d")

        for img_path in class_path.glob("*.ppm"):
            img = cv2.imread(str(img_path))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, img_size)

            images.append(img)
            labels.append(class_id)

    return np.array(images), np.array(labels)

In [7]:
def load_test_data(test_dir, test_labels_csv, img_size):
    test_df = pd.read_csv(test_labels_csv, sep=";")

    images = []
    labels = []

    for _, row in test_df.iterrows():
        img_path = test_dir / row["Filename"]

        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, img_size)

        images.append(img)
        labels.append(row["ClassId"])

    return np.array(images), np.array(labels)

In [8]:
X_all, y_all = load_training_data(TRAIN_DIR, IMG_SIZE)
X_test_raw, y_test = load_test_data(TEST_DIR, TEST_LABELS, IMG_SIZE)

print(X_all.shape)
print(X_test_raw.shape)

(39209, 96, 96, 3)
(12630, 96, 96, 3)


In [9]:
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_all,
    y_all,
    test_size=0.2,
    random_state=42,
    stratify=y_all
)

print(X_train_raw.shape)
print(X_val_raw.shape)

(31367, 96, 96, 3)
(7842, 96, 96, 3)


In [10]:
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

In [11]:
X_train = preprocess_input(X_train_raw.astype("float32"))
X_val = preprocess_input(X_val_raw.astype("float32"))
X_test = preprocess_input(X_test_raw.astype("float32"))

In [12]:
print(X_train.min(), X_train.max())

-1.0 1.0


In [13]:
base_model = MobileNetV2(
    input_shape=(96, 96, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [14]:
mobilenet_model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(43, activation="softmax")
])

In [15]:
mobilenet_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

mobilenet_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_96             │ (None, 3, 3, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 43)             │         5,547 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,427,499 (9.26 MB)

 Trainable params: 169,515 (662.17 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [16]:
history_mobilenet = mobilenet_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)

Epoch 1/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 122s 120ms/step - accuracy: 0.6102 - loss: 1.3019 - val_accuracy: 0.8127 - val_loss: 0.5787
Epoch 2/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 110s 112ms/step - accuracy: 0.7904 - loss: 0.6368 - val_accuracy: 0.8735 - val_loss: 0.4069
Epoch 3/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 110s 112ms/step - accuracy: 0.8343 - loss: 0.4870 - val_accuracy: 0.8933 - val_loss: 0.3361
Epoch 4/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 106s 108ms/step - accuracy: 0.8654 - loss: 0.4028 - val_accuracy: 0.9083 - val_loss: 0.2830
Epoch 5/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 90s 92ms/step - accuracy: 0.8775 - loss: 0.3537 - val_accuracy: 0.9155 - val_loss: 0.2628
Epoch 6/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 66s 68ms/step - accuracy: 0.8899 - loss: 0.3188 - val_accuracy: 0.9178 - val_loss: 0.2511
Epoch 7/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 67s 69ms/step - accuracy: 0.9010 - loss: 0.2831 - val_accuracy: 0.9249 - val_loss: 0.2301
Epoch 8/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 68s 69ms/step - accuracy: 0.9091 - loss: 0

In [17]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

In [18]:
mobilenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [19]:
history_mobilenet_finetune = mobilenet_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)

Epoch 1/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 90s 86ms/step - accuracy: 0.6362 - loss: 1.8966 - val_accuracy: 0.8944 - val_loss: 0.3311
Epoch 2/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 77s 79ms/step - accuracy: 0.7526 - loss: 0.9513 - val_accuracy: 0.8924 - val_loss: 0.3329
Epoch 3/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 78s 80ms/step - accuracy: 0.8028 - loss: 0.6627 - val_accuracy: 0.9075 - val_loss: 0.2905
Epoch 4/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 82s 84ms/step - accuracy: 0.8369 - loss: 0.5169 - val_accuracy: 0.9171 - val_loss: 0.2521
Epoch 5/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 86s 88ms/step - accuracy: 0.8609 - loss: 0.4185 - val_accuracy: 0.9305 - val_loss: 0.2135
Epoch 6/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 95s 97ms/step - accuracy: 0.8810 - loss: 0.3609 - val_accuracy: 0.9387 - val_loss: 0.1888
Epoch 7/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 83s 84ms/step - accuracy: 0.8986 - loss: 0.3015 - val_accuracy: 0.9448 - val_loss: 0.1711
Epoch 8/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 84s 85ms/step - accuracy: 0.9118 - loss: 0.2581 - 

In [20]:
mobilenet_model.save("../models/mobilenetv2_gtsrb_experiment.keras")